In [1]:
import os
import sys
os.chdir('..')
# sys.path.insert('.')

In [2]:
import argparse
import os
import time
import torch
import pandas as pd
from importlib import reload
import json
import numpy as np
from dotenv import load_dotenv
from copy import deepcopy
import glob
import re
import ast
import json
from ast import literal_eval
load_dotenv()

True

In [3]:
os.environ['CUDA_VISIBLE_DEVICES'] = '5'

In [4]:
from transformer_lens import HookedTransformer, HookedTransformerConfig
import pickle
# Automatically select device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
def load_finetuned_model_lens_from_dir(dir: str, device: str = device) -> HookedTransformer:
    """
    Load a fine-tuned TransformerLens model from a specified directory.

    Args:
        dir (str): Directory containing the model files.
        device (str): Device to load the model onto.
    
    Returns:
        HookedTransformer: The loaded TransformerLens model.
    """
    with open(os.path.join(dir, 'model_config.pkl'), 'rb') as f:
        new_cfg_dict = pickle.load(f)
    new_cfg = HookedTransformerConfig.from_dict(new_cfg_dict)
    new_model = HookedTransformer(new_cfg)
    new_model.load_state_dict(torch.load(os.path.join(dir, 'model.pt'), map_location=device))
    return new_model

/raid/home/m13521157/absa-eap-ig/enveap/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Get Counterfacts from Test Data

In [5]:
# List all files in a directory recursively, but stop at the last folder before a file
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list

#### Get the candidates

In [6]:
lang = 'indo'
dataset_folder = 'corrected_splitopinion_typocorrected'
seed = 123
model_path_parent = f'outputs/models/eap/{dataset_folder}/circuit-{lang}_finetune-{lang}/seed_{seed}'
dataset_path = f'hotel_dataset/counterfacts/{dataset_folder}/{lang}_counterfacts.csv'
filtered_data_path = f'test/{lang}_debug_filtered.csv'
full_aos_path = f'test/{lang}_debug_full_aos.csv'
sequence_variants_path = f'test/{lang}_debug_sequence_variant.csv'
eap_output_path = f'test/{lang}_debug_eap_dataset.csv'
langs = ['indo']
counterfact_id = 'counterfactsv3'

In [7]:
files = list_files_recursively(f'outputs/models/eap/{dataset_folder}')
models = [os.path.dirname(f) for f in files if 'topk' not in f]
models = [f for f in models if not f.endswith('full_sft')]
models = list(set(models))
models

['outputs/models/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_777/aos_sequence_variants/full_sft/2025-10-08 01:09:48.770120_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/models/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_123/aos_sequence_variants/full_sft/2025-10-08 01:11:42.891178_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/models/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/full_sft/2025-10-09 01:49:36.368612_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/models/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_9584/aos_sequence_variants/full_sft/2025-10-09 01:49:27.750722_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20',
 'outputs/models/

In [8]:
dataset_dict = {}
for lang in langs:
    with open(f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_train_augmented_noreasoning.json', 'r') as f:
        dataset_dict[lang] = json.load(f)

In [9]:
valid_candidate = {}
for lang in langs:
	valid_candidate[lang] = []
	for idx in range(0, len(dataset_dict[lang]), 5):
		num_of_targets = re.findall(r'\[SSEP\]', dataset_dict[lang][idx]['target'])

		# Initialize the number of triplets deemed valid
		valid_num_of_targets = [0] # List of valid number of targets
		if len(num_of_targets) in valid_num_of_targets and 'null' not in dataset_dict[lang][idx]['target']:
			valid_candidate[lang].append(dataset_dict[lang][idx])

len(valid_candidate['indo'])

191

In [10]:
for lang in langs:
    data_for_csv = {
        'index': [instance['sentence_id'] for instance in valid_candidate[lang]],
        'original_pair': [f"{instance['input']} {instance['target']}" for instance in valid_candidate[lang]],
	}
    df_empty_counterfact = pd.DataFrame(data_for_csv)
    # df_empty_counterfact['corrupted_pair'] = np.nan # Placeholder for corrupted pairs
    df_empty_counterfact['corrupted_pair'] = df_empty_counterfact['original_pair'] # Run this if you want to fill in corrupted pairs later (e.g., filtering first and then filling the corrupted pairs)
    os.makedirs(f'hotel_dataset/{counterfact_id}/{dataset_folder}', exist_ok=True)
    df_empty_counterfact.to_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/{lang}_counterfacts.csv', index=False)

#### Inference

In [11]:
def extract_triplet_fixed(text):
	try:
		matches = list(re.finditer(r"\[([AOS])\]", text))
		if len(matches) >= 3:
			a_start = matches[0].end()
			o_start = matches[1].end()
			s_start = matches[2].end()
			aspect = text[a_start:matches[1].start()].strip()
			opinion = text[o_start:matches[2].start()].strip()
			sentiment = text[s_start:].split()[0].strip()
			return (aspect, opinion, sentiment)
	except:
		return None
		
def format_counterfactuals(input_path):
	df = pd.read_csv(input_path, encoding="utf-8", quoting=1)

	# df = df.dropna(subset=["corrupted_pair"])
	df['original_sentence'] = df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
	temp_column = df['original_pair'].apply(lambda x: x.split('[A] [O] [S]')[-1].strip())
	temp_column = temp_column.apply(lambda x: x.split('[SSEP]')).apply(lambda x: [i.strip() for i in x])
	temp_column = temp_column.apply(lambda x: [extract_triplet_fixed(i) for i in x])
	df['original_triplet'] = deepcopy(temp_column)

	try:
		df['counterfact4_replaced'] = df['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[0].strip())
		temp_column = df['corrupted_pair'].apply(lambda x: x.split('[A] [O] [S]')[-1].strip())
		temp_column = temp_column.apply(lambda x: x.split('[SSEP]'))
		temp_column = temp_column.apply(lambda x: [i.strip() for i in x])
		temp_column = temp_column.apply(lambda x: [extract_triplet_fixed(i) for i in x])
		df['counterfact_triplet4_replaced'] = deepcopy(temp_column)
	except KeyError:
		df['counterfact4_replaced'] = None
		df['counterfact_triplet4_replaced'] = None
		print("KeyError: 'corrupted_pair' is not in a valid format (must be string and no None value). Skipping replacement.")

	df_out = df[['index', 'original_sentence', 'original_triplet', 'counterfact4_replaced', 'counterfact_triplet4_replaced']].copy()
	folder = os.path.dirname(input_path)
	filename = os.path.basename(input_path)
	print(f"Saving formatted data to {os.path.join(folder, f'formatted_{filename}')}")
	df_out.to_csv(os.path.join(folder, f"formatted_{filename}"), index=False)
	print(f"Saved {len(df_out)} rows to {os.path.join(folder, f'formatted_{filename}')}")
	return df_out

def convert_triplet_string(triplet_str: str) -> tuple:
    """
    Safely parses a stringified triplet like '[("aspect", "opinion", "sentiment")]'
    and returns the individual components.

    Returns:
        Tuple of (aspect, opinion, sentiment) or empty strings if invalid.
    """
    try:
        triplet = ast.literal_eval(triplet_str)
        return tuple(triplet)
    except (ValueError, SyntaxError, IndexError):
        return "", "", ""

def filter_correct_data(model, df, input_col, label_col, filter_mode="AOS", filter_only_correct=True, save_path=None, max_tokens=150):

	valid_modes = {"A", "O", "S", "AOS"}
	filter_mode = filter_mode.upper()
	if filter_mode not in valid_modes:
		raise ValueError(f"Invalid filter_mode '{filter_mode}'. Must be one of {valid_modes}.")

	suffix = ' [A] [O] [S]'
	# For testing, take 5 first and 5 last instances of the df
	# df = pd.concat([df.iloc[:5], df.iloc[-5:]]).reset_index(drop=True)
	inputs = df[input_col].tolist()
	labels = df[label_col].tolist()

	results, expected_labels, match_flags = [], [], []

	for prompt, label_raw in zip(inputs, labels):
		full_prompt = prompt + suffix
		print(f"Processing prompt: {full_prompt}")
		output = model.generate(
			input=full_prompt,
			max_new_tokens=max_tokens,
			stop_at_eos=True,
			do_sample=False,
			return_type="str"
		)

		# Remove special tokens and clean up
		raw_output = re.sub(r"<\|endoftext\|>", "", output)

		# Handle multiple triplets
		is_match = True
		triplets_str = raw_output.split("[A] [O] [S]")[-1].strip()
		triplets_str_temp = triplets_str.split("[SSEP]")
		# print(f"Triplets string: {triplets_str}")
		triplets_str_temp = [i.strip() for i in triplets_str_temp]
		labels = convert_triplet_string(label_raw)
		# print(f'Labels: {labels}')
		for triplet_str in triplets_str_temp:
			triplet = extract_triplet_fixed(triplet_str)
			if triplet not in labels:
				print(f"Mismatch found: {triplet} not in {labels}")
				is_match = False
				break
		
		formatted_labels = [f"[A] {label[0]} [O] {label[1]} [S] {label[2]}" for label in labels]
		formatted_labels = " [SSEP] ".join(formatted_labels)
		if is_match:
			results.append(formatted_labels) # Same ordering as the formatted labels
		else:
			results.append(triplets_str)
		expected_labels.append(formatted_labels)
		match_flags.append(is_match)

	df_result = df.copy()
	df_result["original_label"] = expected_labels
	df_result["inference"] = results
	df_result["is_match"] = match_flags

	total = len(df_result)
	correct = df_result["is_match"].sum()
	print(f"Correct: {correct} / {total} ({correct / total:.2%}) with mode [{filter_mode}]")

	if filter_only_correct:
		df_result = df_result[df_result["is_match"]].reset_index(drop=True)
		
	if save_path:
		df_result.to_csv(save_path, index=False)
	
	return df_result

In [12]:
test_path = 'hotel_dataset/test_counterfacts/indo_counterfacts.csv'
df_counterfact_test = pd.read_csv(test_path)
format_counterfactuals(test_path)
df_counterfact_test = pd.read_csv(os.path.dirname(test_path) + f"/formatted_{os.path.basename(test_path)}")
df_counterfact_test

Saving formatted data to hotel_dataset/test_counterfacts/formatted_indo_counterfacts.csv
Saved 70 rows to hotel_dataset/test_counterfacts/formatted_indo_counterfacts.csv


,index,original_sentence,original_triplet,counterfact4_replaced,counterfact_triplet4_replaced
0,2501,pelayanan lumayan baik .,"[('pelayanan', 'lumayan baik', 'positive')]",satelit sangat wangi .,"[('satelit', 'sangat wangi', 'negative')]"
1,2506,suasananya kurang nyaman . untuk menginap haru...,"[('suasananya', 'kurang nyaman', 'negative')]",keretanya cukup ramah . untuk menginap harus d...,"[('keretanya', 'cukup ramah', 'positive')]"
2,2512,makanan yang kurang memuaskan .,"[('makanan', 'kurang memuaskan', 'negative')]",sepeda yang berwangi bunga .,"[('sepeda', 'berwangi bunga', 'positive')]"
3,2516,tempatnya nyaman untuk istirahat .,"[('tempatnya', 'nyaman', 'positive')]",meja nya ribut untuk istirahat .,"[('meja nya', 'ribut', 'negative')]"
4,2517,kamar bersih .,"[('kamar', 'bersih', 'positive')]",daun bau .,"[('daun', 'bau', 'negative')]"
...,...,...,...,...,...
65,2618,hotelnya bagus bersih nyaman .,"[('hotelnya', 'bagus', 'positive'), ('hotelnya...",kapal galak malu kasar .,"[('kapal', 'galak', 'negative'), ('hotelnya', ..."
66,2696,tempatnya bersih dan nyaman . staf nya ramah .,"[('tempatnya', 'bersih', 'positive'), ('tempat...",kantong gelap dan curang . tombolnya lama .,"[('kantong', 'gelap', 'negative'), ('kantong',..."
67,2994,kamar bagus dan bersih tetapi kurang besar .,"[('kamar', 'bagus', 'positive'), ('kamar', 'be...",buku galak dan judes tetapi sangat kuat .,"[('buku', 'galak', 'negative'), ('buku', 'jude..."
68,3015,"staf kurang ramah , sarung bantal kotor , pint...","[('staf', 'kurang ramah', 'negative'), ('sarun...","penghapus manis sekali , papan tulis gagah , t...","[('penghapus', 'manis sekali', 'positive'), ('..."


In [ ]:
# This is for debugging purpose (only one model not several)
model_path = None
for temp in models:
	if 'indo' in temp:
		model_path = temp
		break
model = load_finetuned_model_lens_from_dir(model_path)
device = (
	torch.device("mps") if torch.backends.mps.is_available()
	else torch.device("cuda") if torch.cuda.is_available()
	else torch.device("cpu")
)
model.to(device)
model.eval()

print(model_path)

In [ ]:
# This is for debugging purpose (only one model not several)
filtered_data_path = f'temp/debug_for_multitriplet.csv'
filtered_df = filter_correct_data(
	model,
	df_counterfact_test,
	"original_sentence",
	"original_triplet",
	filter_mode="AOS",
	filter_only_correct=False,
	save_path=filtered_data_path,
	max_tokens=150  # Adjusted max_tokens to 150 for better performance
)

In [ ]:
filtered_df

In [ ]:
false_instance = filtered_df.loc[filtered_df['index'] == 2900, :]
target = false_instance['original_label'].values[0]
pred = false_instance['inference'].values[0]
print(f"Target: {target}")
print(f"Prediction: {pred}")
print(f"Match: {target == pred}")

In [13]:
filtered_dfs = {}
for model_path in models:
	model = load_finetuned_model_lens_from_dir(model_path)
	device = (
		torch.device("mps") if torch.backends.mps.is_available()
		else torch.device("cuda") if torch.cuda.is_available()
		else torch.device("cpu")
	)
	model.to(device)
	model.eval()

	print(model_path)

	if 'indo' in model_path:
		dataset_path = f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts.csv'
		language = 'indo'
	elif 'eng' in model_path:
		dataset_path = f'hotel_dataset/{counterfact_id}/{dataset_folder}/eng_counterfacts.csv'
		language = 'eng'
	elif 'sunda' in model_path:
		dataset_path = f'hotel_dataset/{counterfact_id}/{dataset_folder}/sunda_counterfacts.csv'
		language = 'sunda'
	else:
		raise ValueError("Unknown model language in path: " + model_path)
	
	df = pd.read_csv(dataset_path)
	print(f"Dataset loaded from {dataset_path} ({len(df)} rows)")
	if "original_pair" in df.columns:
		format_counterfactuals(dataset_path)
		folder = os.path.dirname(dataset_path)
		filename = os.path.basename(dataset_path)
		formated_path = os.path.join(folder, f"formatted_{filename}")
		df = pd.read_csv(formated_path)

	seed = model_path.split('/')[5]
	filtered_data_path = f'temp/{dataset_folder}/{language}_{seed}.csv'
	os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
	id = filtered_data_path.split('/')[-1]
	filtered_df = filter_correct_data(
		model,
		df,
		"original_sentence",
		"original_triplet",
		filter_mode="AOS",
		filter_only_correct=True,
		save_path=filtered_data_path,
		max_tokens=150  # Adjusted max_tokens to 150 for better performance
	)
	filtered_dfs[id] = filtered_df.copy()

Moving model to device:  cuda
outputs/models/eap/corrected_splitopinion_typocorrected/circuit-indo_finetune-indo/seed_777/aos_sequence_variants/full_sft/2025-10-08 01:09:48.770120_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20
Dataset loaded from hotel_dataset/counterfactsv3/corrected_splitopinion_typocorrected/indo_counterfacts.csv (191 rows)
Saving formatted data to hotel_dataset/counterfactsv3/corrected_splitopinion_typocorrected/formatted_indo_counterfacts.csv
Saved 191 rows to hotel_dataset/counterfactsv3/corrected_splitopinion_typocorrected/formatted_indo_counterfacts.csv
Processing prompt: tidak dapat snack . setelah di keluhan , baru dikasik snacknya . [A] [O] [S]


  9%|▊         | 13/150 [00:01<00:17,  7.78it/s]


Processing prompt: kamarnya oke . [A] [O] [S]


 10%|█         | 15/150 [00:00<00:04, 29.04it/s]


Processing prompt: tempat tdr kurang bersih . [A] [O] [S]


...

Processing prompt: wifi kurang joss . [A] [O] [S]


 10%|█         | 15/150 [00:00<00:04, 29.37it/s]


Processing prompt: sayang , air panasnya tidak terlalu panas , jadi kalau mandi kedinginan . hhehe . [A] [O] [S]


 20%|██        | 30/150 [00:00<00:03, 30.23it/s]

Correct: 183 / 191 (95.81%) with mode [AOS]


In [14]:
print('test')

test


In [15]:
filtered_dfs = {}
results_path = os.listdir(f'temp/{dataset_folder}')
for path in results_path:
    filtered_dfs[path] = pd.read_csv(os.path.join(f'temp/{dataset_folder}', path))
print(filtered_dfs.keys())
print(len(filtered_dfs))

dict_keys(['indo_seed_9584.csv', 'indo_seed_123.csv', 'indo_seed_31415.csv', 'indo_seed_2024.csv', 'indo_seed_777.csv'])
5


In [16]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

# Convert to list and sort
indexes = sorted(list(indexes))
len(indexes)

171

In [18]:
df_check = pd.read_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilled.csv')
df_check

,index,original_pair,corrupted_pair,num_of_triplets
0,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,2
1,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,2
2,80,tv nya saja yang tidak bagus karena siaran tvn...,tv nya saja yang tidak bagus karena siaran tvn...,2
3,134,pelayanannya ramah . [A] [O] [S] [A] pelayanan...,pelayanannya ramah . [A] [O] [S] [A] pelayanan...,2
4,136,tempat parkir mobil yang terbatas . [A] [O] [S...,tempat parkir mobil yang terbatas . [A] [O] [S...,2
...,...,...,...,...
70,2315,dpet kamar yang kurang menarik . semoga next d...,dpet kamar yang kurang menarik . semoga next d...,2
71,2391,kamarnya benar benar sesuai dengan yang ada di...,kamarnya benar benar sesuai dengan yang ada di...,2
72,2393,"tempat transit , makanannya enak . [A] [O] [S]...","tempat transit , makanannya enak . [A] [O] [S]...",2
73,2409,tidak ada lift . kesulitannya hanya mengangkut...,tidak ada lift . kesulitannya hanya mengangkut...,2


In [22]:
df_check['false_indexes'] = df_check['index'].isin(indexes)
df_check[~df_check['false_indexes']]

,index,original_pair,corrupted_pair,num_of_triplets,false_indexes
11,267,kurang cuma pada sarapan yang cuma di kasih 1 ...,kurang cuma pada sarapan yang cuma di kasih 1 ...,2,False
29,932,selimut/sprey kurang bersih . [A] [O] [S] [A] ...,selimut/sprey kurang bersih . [A] [O] [S] [A] ...,2,False
47,1489,televisi nyaa buram . [A] [O] [S] [A] televisi...,televisi nyaa buram . [A] [O] [S] [A] televisi...,2,False
66,2091,hotel airy terbaik yang pernah saya temui . [A...,hotel airy terbaik yang pernah saya temui . [A...,2,False


In [17]:
dataset_folder

'corrected_splitopinion_typocorrected'

In [40]:
original_df = pd.read_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts.csv')
original_df = original_df[original_df['index'].isin(indexes)].reset_index(drop=True)
original_df['num_of_triplets'] = original_df['original_pair'].apply(lambda x: len(re.findall(r'\[SSEP\]', x)) + 1)
original_df

,index,original_pair,corrupted_pair,num_of_triplets
0,4,"tidak dapat snack . setelah di keluhan , baru ...","tidak dapat snack . setelah di keluhan , baru ...",1
1,10,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,kamarnya oke . [A] [O] [S] [A] kamarnya [O] ok...,1
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,1
3,39,2 kali inap di situ dengan kamar yang berbeda ...,2 kali inap di situ dengan kamar yang berbeda ...,1
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,1
...,...,...,...,...
173,2458,karena terlalu banyak kamar jadi tidak diperha...,karena terlalu banyak kamar jadi tidak diperha...,1
174,2462,wifi lebih baik dijangkau dalam kamar . yang l...,wifi lebih baik dijangkau dalam kamar . yang l...,1
175,2463,tidak ada wifi sesuai yang di iklankan . [A] [...,tidak ada wifi sesuai yang di iklankan . [A] [...,1
176,2495,wifi kurang joss . [A] [O] [S] [A] wifi [O] ku...,wifi kurang joss . [A] [O] [S] [A] wifi [O] ku...,1


In [ ]:
# Select 50 rows randomly with seed from original_df, returning a new dataframe
sampled_df = original_df.sample(n=75, random_state=42).sort_values(by='index')
sampled_df

,index,original_pair,corrupted_pair,num_of_triplets
2,18,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,tempat tdr kurang bersih . [A] [O] [S] [A] tem...,2
4,54,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,tidak ada sarapan . [A] [O] [S] [A] sarapan [O...,2
6,80,tv nya saja yang tidak bagus karena siaran tvn...,tv nya saja yang tidak bagus karena siaran tvn...,2
9,134,pelayanannya ramah . [A] [O] [S] [A] pelayanan...,pelayanannya ramah . [A] [O] [S] [A] pelayanan...,2
10,136,tempat parkir mobil yang terbatas . [A] [O] [S...,tempat parkir mobil yang terbatas . [A] [O] [S...,2
...,...,...,...,...
167,2315,dpet kamar yang kurang menarik . semoga next d...,dpet kamar yang kurang menarik . semoga next d...,2
169,2391,kamarnya benar benar sesuai dengan yang ada di...,kamarnya benar benar sesuai dengan yang ada di...,2
170,2393,"tempat transit , makanannya enak . [A] [O] [S]...","tempat transit , makanannya enak . [A] [O] [S]...",2
171,2409,tidak ada lift . kesulitannya hanya mengangkut...,tidak ada lift . kesulitannya hanya mengangkut...,2


In [48]:
# Save to CSV
sampled_df.to_csv(f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilled.csv', index=False)

In [49]:
f'hotel_dataset/{counterfact_id}/{dataset_folder}/indo_counterfacts_tobefilled.csv'

'hotel_dataset/counterfactsv3/corrected_splitopinion_typocorrected/indo_counterfacts_tobefilled.csv'

#### Check from the old filled counterfacts (only for counterfactv2)

In [ ]:
checkdf = pd.read_csv('hotel_dataset/counterfactsv2.1/clean_traintruncated/indo_counterfacts.csv')
# indexes_checkdf = checkdf.iloc[50:]['index']
indexes_checkdf = checkdf['index']
len(indexes_checkdf)

In [ ]:
checkdf

In [ ]:
set(indexes_checkdf) - set(indexes)

In [ ]:
print(set(indexes) - set(indexes_checkdf))

In [ ]:
for lang in ['indo', 'eng', 'sunda']:
	empty_counterfacts_path = f'hotel_dataset/empty_counterfactsv2/{dataset_folder}/{lang}_counterfacts.csv'
	df_empty_counterfacts = pd.read_csv(empty_counterfacts_path)
	df_empty_counterfacts = df_empty_counterfacts[df_empty_counterfacts['index'].isin(indexes)].reset_index(drop=True)
	df_empty_counterfacts['num_of_targets'] = df_empty_counterfacts['original_pair'].apply(lambda x: 1 + len(re.findall(r'\[SSEP\]', x)))
	df_empty_counterfacts.to_csv(os.path.join(os.path.dirname(empty_counterfacts_path), f'{lang}_counterfacts_tobefilled.csv'), index=False)

#### Test from results

In [ ]:
results_paths = list_files_recursively('outputs/evals/eap/clean_traintruncated')
results_paths = [f for f in results_paths if 'topk' not in f and 'inference_results.json' in f]
results_paths

In [ ]:
indexes_true = {}
for result_path in results_paths:
	with open(result_path, 'r') as f:
		results = json.load(f)
	seed = result_path.split('/')[5].split('_')[-1]
	lang = result_path.split('/')[4].split('_')[0].split('-')[-1]
	indexes_true[f'{lang}_{seed}'] = []
	for idx in range(0, len(results), 5):
		is_match = True
		parsed_target_list = [extract_triplet_fixed(i) for i in results[idx]['target_list']]
		# print(results[idx]['target_list'])
		# print(results[idx]['prediction_list'])
		for pred in results[idx]['prediction_list']:
			parsed_pred = extract_triplet_fixed(pred)
			if parsed_pred not in parsed_target_list:
				is_match = False
				break
			
		if is_match:
			indexes_true[f'{lang}_{seed}'].append(results[idx]['sentence_id'])

In [ ]:
indexes_intersect = set()
for key, indexes in indexes_true.items():
	# Intersect all indexes to the indexes_intersect set
	if not indexes_intersect:
		indexes_intersect = set(indexes)
	else:
		indexes_intersect.intersection_update(indexes)
len(indexes_intersect)

#### Debug create aos dataset

In [ ]:
model_path = None
for temp in models:
	if 'indo' in temp:
		model_path = temp
		break
model = load_finetuned_model_lens_from_dir(model_path)
device = (
	torch.device("mps") if torch.backends.mps.is_available()
	else torch.device("cuda") if torch.cuda.is_available()
	else torch.device("cpu")
)
model.to(device)
model.eval()

print(model_path)

In [ ]:
raw_data_path = 'hotel_dataset/test_counterfacts/indo_counterfacts.csv'
format_counterfactuals(raw_data_path)
formatted_data_path = 'hotel_dataset/test_counterfacts/formatted_indo_counterfacts.csv'
df_test_formatted = pd.read_csv(formatted_data_path)
df_test_formatted

In [ ]:
filtered_data_path = os.path.join(os.path.dirname(formatted_data_path), f'filtered_indo_counterfacts.csv')
os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
filtered_df = filter_correct_data(
	model,
	df_test_formatted,
	"original_sentence",
	"original_triplet",
	filter_mode="AOS",
	filter_only_correct=False,
	save_path=filtered_data_path,
	max_tokens=150  # Adjusted max_tokens to 150 for better performance
)

In [ ]:
filtered_df

In [ ]:
def create_sequences(a, o, s):
	return {
		"AOS": f"[A] {a} [O] {o} [S] {s}",
		"ASO": f"[A] {a} [S] {s} [O] {o}",
		"SAO": f"[S] {s} [A] {a} [O] {o}",
		"OAS": f"[O] {o} [A] {a} [S] {s}",
		"OSA": f"[O] {o} [S] {s} [A] {a}",
	}

def create_aos_sequence_variant(dataset_path):

	df = pd.read_csv(dataset_path)
	records_with_match = []
	
	for _, row in df.iterrows():
		original_triplet = literal_eval(row['original_triplet'])
		counterfact_triplet = literal_eval(row['counterfact_triplet4_replaced'])
		assert len(original_triplet) == len(counterfact_triplet), "Original and counterfactual triplets must have the same length."
		assert len(original_triplet) > 0, "Original triplet must not be empty."
		assert len(counterfact_triplet) > 0, "Counterfactual triplet must not be empty."

		orig_seq_list = {"AOS": [], "ASO": [], "SAO": [], "OAS": [], "OSA": []}
		cf_seq_list = {"AOS": [], "ASO": [], "SAO": [], "OAS": [], "OSA": []}
		for i, triplet in enumerate(original_triplet):
			orig_a, orig_o, orig_s = triplet
			cf3_a, cf3_o, cf3_s = counterfact_triplet[i]
			orig_seq = create_sequences(orig_a, orig_o, orig_s)
			cf_seq = create_sequences(cf3_a, cf3_o, cf3_s)
			for order, seq in orig_seq.items():
				orig_seq_list[order].append(seq)
				cf_seq_list[order].append(cf_seq[order])

		for order, orig_seqs in orig_seq_list.items():
			final_orig_seq = " [SSEP] ".join(orig_seqs)
			final_cf_seq = " [SSEP] ".join(cf_seq_list[order])
			records_with_match.append({
				"order": order,
				"original_sentence": row["original_sentence"],
				"original_triplet": row["original_triplet"],
				"original_label_variant": final_orig_seq,
				"counterfact4_replaced": row["counterfact4_replaced"],
				"counterfact_triplet4_replaced": row["counterfact_triplet4_replaced"],
				"counterfact_label_variant": final_cf_seq,
				"is_match": row["is_match"]
			})

	return pd.DataFrame(records_with_match)

In [ ]:
df_aos_sequence_variant = create_aos_sequence_variant(filtered_data_path)
df_aos_sequence_variant

In [ ]:
from transformer_lens import HookedTransformer

def get_tag_suffix(order):
    return " ".join(f"[{ch}]" for ch in order)

def build_eap_dataset(
    model: HookedTransformer,
    df: pd.DataFrame,
    sentence_col: str,
    triplet_col: str,
    corrupted_col: str,
    corrupted_triplet_col: str,
    suffix: str,
    filer_same_length_counterfactuals: bool = True,
    append_labels= False,
) -> pd.DataFrame:
    """
    Builds an EAP dataset from a filtered dataframe for use with EAP-IG,
    saving both token ids and string values of correct/incorrect labels.

    Args:
        model (HookedTransformer): TransformerLens model used for tokenization.
        df (pd.DataFrame): DataFrame containing sentence and counterfactual data.
        sentence_col (str): Column name for the original sentence.
        triplet_col (str): Column name for the original triplet string.
        corrupted_col (str): Column name for the corrupted sentence.
        corrupted_triplet_col (str): Column name for the corrupted triplet string.
        suffix (str): Prompt suffix to add (e.g., "[A]"). Use "[A] [O] [S]" for the full simultaneous AOS dataset.
        filer_same_length_counterfactuals (bool): If True, remove datapoints where
            counterfactual token length differs from original token length.

    Returns:
        pd.DataFrame: DataFrame with clean/corrupted prompts, token indices, and raw label texts.
    """

    if suffix == "[A]":
        idx = 0
    elif suffix == "[O]":
        idx = 1
    elif suffix == "[S]":
        idx = 2
    elif suffix == "[A] [O] [S]":
        assert "order" in df.columns, "Column 'order' must exist in the DataFrame"
        idx = False
    else:
        raise ValueError(f"Invalid suffix '{suffix}'. Must be one of '[A]', '[O]', '[S]' or '[A] [O] [S]'.")

    eap_data = []
    num_removed = 0
    for _, row in df.iterrows():
        if row["is_match"] and type(row[corrupted_col]) == str:

            if not idx:
                suffix = get_tag_suffix(row["order"])

            clean = row[sentence_col] + f" {suffix}"
            corrupted = row[corrupted_col] + f" {suffix}"
            print(f"Processing clean: {clean}")
            print(f"Processing corrupted: {corrupted}")

            if filer_same_length_counterfactuals:
                clean_tokens = model.to_tokens(clean)
                corrupted_tokens = model.to_tokens(corrupted)
                if clean_tokens.shape[1] != corrupted_tokens.shape[1]:
                    num_removed += 1
                    continue

            try:
                if not idx:
                    correct_label = row[triplet_col]
                    incorrect_label = row[corrupted_triplet_col]
                
                else: # TODO: Cannot yet handle multiple triplets in the same row
                    original_triplet = ast.literal_eval(row[triplet_col])
                    corrupted_triplet = ast.literal_eval(row[corrupted_triplet_col])
                    
                    correct_label = original_triplet[0][idx]
                    incorrect_label = corrupted_triplet[0][idx]

                correct_idx = model.to_tokens(f" {correct_label}").tolist()
                incorrect_idx = model.to_tokens(f" {incorrect_label}").tolist()

                if len(correct_idx[0]) != len(incorrect_idx[0]):
                    num_removed += 1
                    continue

                # # if label has multiple tokens, append all but last label tokens to clean and corrupted
                # # because we need to obtain the right logit conditioned on the correct prefix
                # if append_labels:
                #     if len(correct_idx[0]) > 1: # TODO
                #         clean += model.to_string(correct_idx[0][:-1])
                #         corrupted += model.to_string(incorrect_idx[0][:-1])

                eap_data.append({
                    "clean": clean,
                    "corrupted": corrupted,
                    "correct_label": correct_label,
                    "incorrect_label": incorrect_label,
                    "correct_idx": correct_idx,
                    "incorrect_idx": incorrect_idx,
                })
            except Exception as e:
                print(f"Skipping row due to parsing/tokenizing error: {e}")
                continue

    if filer_same_length_counterfactuals:
        print(f"Removed {num_removed} out of {len(df)} datapoints that does not match token length.")
    print(f"Filtered data size {len(eap_data)=}")
    
    eap_df = pd.DataFrame(eap_data)
    eap_df["correct_idx"] = eap_df["correct_idx"].apply(str)
    eap_df["incorrect_idx"] = eap_df["incorrect_idx"].apply(str)    
    return eap_df

In [ ]:
model.to_string([508, 32, 60, 281, 6895, 28618, 508, 46, 60, 40163, 43857, 79187, 508, 50, 60, 6785])

In [ ]:
df_eap_output = build_eap_dataset(
    model,
    df_aos_sequence_variant,
	sentence_col="original_sentence",
	triplet_col="original_label_variant",
	corrupted_col="counterfact4_replaced",
	corrupted_triplet_col="counterfact_label_variant",
    suffix="[A] [O] [S]",
    filer_same_length_counterfactuals=True,
    append_labels=True,
)

In [ ]:
df_eap_output

### Store Temporary Counterfacts

In [ ]:
counterfacts_dict = {}
for lang in ['eng', 'indo', 'sunda']:
	new_counterfact_path = f'../hotel_dataset/empty_counterfacts/{lang}_counterfacts_clean_traintruncated.csv'
	old_counterfact_path = f'../hotel_dataset/empty_counterfacts/{lang}_counterfacts.csv'

	counterfacts_dict[lang] = {
		'new': pd.read_csv(new_counterfact_path),
		'old': pd.read_csv(old_counterfact_path)
	}

In [ ]:
import pandas as pd

def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df

google_sheet_id = '1cukGqysonkhQFD8_0rWgLYpMg95EPqm_5B3svOryf0A' 
gid_eng = '1640284131'
gid_indo = '1378695901'
gid_sunda = '354418125'
filled_counterfacts = {}
try:
	for lang in ['eng', 'indo', 'sunda']:
		filled_counterfacts[lang] = get_google_sheet(google_sheet_id, globals()[f'gid_{lang}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

In [ ]:
filled_counterfacts['sunda']

In [ ]:
for lang in ['eng', 'indo', 'sunda']:
	df_new = counterfacts_dict[lang]['new']
	df_filled = filled_counterfacts[lang]
	df_new['notes'] = np.nan
	df_new['difficult_words'] = np.nan
	df_new['valid'] = np.nan

	# Get all df_filled rows that the index column are in df_new
	df_filtered = df_filled[df_filled['index'].isin(df_new['index'])]

	# Fill the corrupted_pair, notes, difficult_words, and valid in the df_new with the corresponding value in df_filtered
	df_new['corrupted_pair'] = df_new['index'].map(df_filtered.set_index('index')['corrupted_pair'])
	df_new['notes'] = df_new['index'].map(df_filtered.set_index('index')['notes'])
	df_new['difficult_words'] = df_new['index'].map(df_filtered.set_index('index')['difficult_words'])
	df_new['valid'] = df_new['index'].map(df_filtered.set_index('index')['valid'])

In [ ]:
# Fill all the missing value in valid column with False
for lang in ['eng', 'indo', 'sunda']:
	df_new = counterfacts_dict[lang]['new']
	df_new['valid'] = df_new['valid'].fillna(False)

In [ ]:
counterfacts_dict['indo']['new'].info()

In [ ]:
# Store in csv
for lang in ['eng', 'indo', 'sunda']:
	df_new = counterfacts_dict[lang]['new']
	df_new.to_csv(f'../hotel_dataset/empty_counterfacts/{lang}_counterfacts_clean_traintruncated_filled.csv', index=False)

### Merge Counterfacts

In [ ]:
import pandas as pd

def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df

google_sheet_id = '1cukGqysonkhQFD8_0rWgLYpMg95EPqm_5B3svOryf0A' 
gid_eng = '1060013247'
gid_indo = '1640463339'
gid_sunda = '1701198029'
filled_counterfacts = {}
try:
	for lang in ['eng', 'indo', 'sunda']:
		filled_counterfacts[lang] = get_google_sheet(google_sheet_id, globals()[f'gid_{lang}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

In [ ]:
valid_indexes = (filled_counterfacts['sunda']['valid'] & filled_counterfacts['indo']['valid'] & filled_counterfacts['eng']['valid'])

In [ ]:
# Take only the valid indexes from the filled_counterfacts
for lang in ['eng', 'indo', 'sunda']:
    filled_counterfacts[lang].loc[valid_indexes, ['index', 'original_pair', 'corrupted_pair']].to_csv(f'../hotel_dataset/counterfacts/clean_traintruncated/{lang}_counterfacts.csv')

In [ ]:
import re
def extract_triplet_fixed(text):
	try:
		matches = list(re.finditer(r"\[([AOS])\]", text))
		if len(matches) >= 6:
			a_start = matches[3].end()
			o_start = matches[4].end()
			s_start = matches[5].end()
			aspect = text[a_start:matches[4].start()].strip()
			opinion = text[o_start:matches[5].start()].strip()
			sentiment = text[s_start:].split()[0].strip()
			return [(aspect, opinion, sentiment)]
	except:
		return None

In [ ]:
text_test = "the hot shower isn't working . [A] [O] [S] [A] hot shower [O] isn't working [S] negative"
extract_triplet_fixed(text_test)

In [ ]:
df_test = pd.read_csv('../test/eng_debug_sequence_variant.csv')

In [ ]:
from ast import literal_eval
literal_eval(df_test.loc[372, 'original_triplet'])

### Use Subset of Counterfact Dataset

In [ ]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df

google_sheet_id = '1cukGqysonkhQFD8_0rWgLYpMg95EPqm_5B3svOryf0A' 
gid_eng = '1640284131'
gid_indo = '905883810'
gid_sunda = '354418125'
filled_counterfacts = {}
try:
	for lang in ['eng', 'indo', 'sunda']:
		filled_counterfacts[lang] = get_google_sheet(google_sheet_id, globals()[f'gid_{lang}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

In [ ]:
filled_counterfacts['indo']['original_pair'] = filled_counterfacts['indo']['original_pair'].apply(lambda x: x.strip())
filled_counterfacts['indo']['corrupted_pair'] = filled_counterfacts['indo']['corrupted_pair'].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [ ]:
filled_counterfacts['indo']

In [ ]:
os.makedirs('hotel_dataset/counterfactsv2.1/clean_traintruncated', exist_ok=True)
filled_counterfacts['indo'].loc[filled_counterfacts['indo']['valid'], ['index', 'original_pair', 'corrupted_pair']].to_csv('hotel_dataset/counterfactsv2.1/clean_traintruncated/indo_counterfacts.csv', index=False)

### Format Eval Results

In [2]:
# Recursively list all json files an a directory
import os
import glob
def list_json_files(directory):
	"""
	Recursively lists all JSON files in a directory.

	Args:
		directory (str): The directory to search for JSON files.

	Returns:
		list: A list of paths to JSON files.
	"""
	return glob.glob(os.path.join(directory, '**', '*.json'), recursive=True)

In [3]:
eval_results_topk = list_json_files('outputs/evalsv2.1/eap/clean_traintruncated/circuit-indo_finetune-indo')
eval_results_topk = [path for path in eval_results_topk if 'inference_results.json' in path]
eval_results_topk

['outputs/evalsv2.1/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/topk_5000/2025-08-06 11:31:19.958432_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_topk-5000/2025-08-06 11:31:19.958432_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_topk-5000/inference_results.json',
 'outputs/evalsv2.1/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/topk_2000/2025-08-06 06:53:43.439169_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_topk-2000/2025-08-06 06:53:43.439169_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_topk-2000/inference_results.json',
 'outputs/evalsv2.1/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/topk_1000/2025-08-06 02:16:12.858095_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_l

In [4]:
eval_results_full = list_json_files('outputs/evals/eap/clean_traintruncated/circuit-indo_finetune-indo')
eval_results_full = [eval_result for eval_result in eval_results_full if 'inference_results.json' in eval_result]
eval_results_full = [eval_result for eval_result in eval_results_full if 'topk' not in eval_result]
eval_results_full

['outputs/evals/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_31415/aos_sequence_variants/2025-07-21 21:38:33.599204_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-07-21 21:38:33.599204_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/inference_results.json',
 'outputs/evals/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_777/aos_sequence_variants/2025-07-21 21:38:32.383629_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-07-21 21:38:32.383629_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/inference_results.json',
 'outputs/evals/eap/clean_traintruncated/circuit-indo_finetune-indo/seed_9584/aos_sequence_variants/2025-07-25 09:11:22.089150_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20/2025-07-25 09:11:22.089150_tflens_hotel_aste_train_augmented

In [5]:
seeds = [31415, 777, 9584, 123, 2024]

In [6]:
results = {}
for seed in seeds:
	results[seed] = {}
	for eval_result_path in eval_results_topk:
		with open(eval_result_path, 'r') as f:
			result = json.load(f)
		if 'topk' in eval_result_path:
			topk = eval_result_path.split('/')[-2].split('_')[-1]
			results[seed][topk] = deepcopy(result)
	for eval_result_path in eval_results_full:
		with open(eval_result_path, 'r') as f:
			result = json.load(f)
		results[seed]['full'] = deepcopy(result)

In [7]:
results[777].keys()

dict_keys(['topk-5000', 'topk-2000', 'topk-1000', 'full'])

In [ ]:
data_dict = {}
for seed in seeds:
	for topk in results[seed].keys():
		if topk == 'full':
			continue

		data_dict[seed] = {
			'sentence_id': [],
			'element_order': [],
			'input': [],
			'targets': [],
			'predictions_full': [],
			f'predictions_{topk}': []
		}

		for instance in results[seed][topk]:
			data_dict[seed]['sentence_id'].append(instance['sentence_id'])
			data_dict[seed]['element_order'].append(instance['element_order'])
			data_dict[seed]['input'].append(instance['input'])
			data_dict[seed]['targets'].append('\n'.join(instance['target_list']))
			data_dict[seed]['predictions_full'].append('\n'.join(instance['prediction_list']))

		for instance in results[seed][f'{topk}']:
			data_dict[seed][f'predictions_{topk}'].append('\n'.join(instance['prediction_list']))
		
		df_eval = pd.DataFrame(data_dict[seed])
		df_eval.to_csv(f'temp/error_analysis_eval/indo/indov2.1_seed-{seed}_{topk}.csv', index=False)

In [15]:
data_dict_only_aos = {}
for seed in seeds:
	for topk in results[seed].keys():
		if topk == 'full':
			continue

		data_dict_only_aos[seed] = {
			'sentence_id': [],
			'element_order': [],
			'input': [],
			'targets': [],
			'predictions_full': [],
			f'predictions_{topk}': []
		}

		for i, instance in enumerate(results[seed][topk]):
			if i % 5 != 0:
				continue
			data_dict_only_aos[seed]['sentence_id'].append(instance['sentence_id'])
			data_dict_only_aos[seed]['element_order'].append(instance['element_order'])
			data_dict_only_aos[seed]['input'].append(instance['input'])
			data_dict_only_aos[seed]['targets'].append('\n'.join(instance['target_list']))
			data_dict_only_aos[seed]['predictions_full'].append('\n'.join(instance['prediction_list']))

		for i, instance in enumerate(results[seed][f'{topk}']):
			if i % 5 != 0:
				continue
			data_dict_only_aos[seed][f'predictions_{topk}'].append('\n'.join(instance['prediction_list']))
		print(f"Length of each column in seed {seed}:")
		for key, value in data_dict_only_aos[seed].items():
			print(f"{key}: {len(value)}")
		df_eval = pd.DataFrame(data_dict_only_aos[seed])
		df_eval.to_csv(f'temp/error_analysis_eval//indo/indov2.1_seed-{seed}_{topk}_aosonly.csv', index=False)

Length of each column in seed 31415:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-5000: 1000
Length of each column in seed 31415:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-2000: 1000
Length of each column in seed 31415:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-1000: 1000
Length of each column in seed 777:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-5000: 1000
Length of each column in seed 777:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-2000: 1000
Length of each column in seed 777:
sentence_id: 1000
element_order: 1000
input: 1000
targets: 1000
predictions_full: 1000
predictions_topk-1000: 1000
Length of each column in seed 9584:
sentence_id: 1000
element_order: 1000
input: 1000
targets:

In [ ]:
df_eval = pd.DataFrame(data_dict)
df_eval

In [ ]:
df_eval.to_csv('temp/error_analysis_eval/indov2_seed-31415.csv', index=False)

### Get Training Data For Correction

In [11]:
dataset_folder = 'clean_traintruncated'
langs = ['eng', 'indo', 'sunda']
for lang in langs:
	dataset_path = f'hotel_dataset/{lang}/{dataset_folder}/hotel_aste_train_augmented_noreasoning.json'
	with open(dataset_path, 'r') as f:
		data = json.load(f)
		print(f"Loaded {len(data)} records from {dataset_path}")
	filtered_data = []
	for instance in data:
		if instance['instance_id'] % 5 == 0:
			filtered_data.append(instance)
	for i in range(len(filtered_data)):
		target_list = filtered_data[i]['target'].split(' [SSEP] ')
		filtered_data[i]['target_list'] = target_list.copy()
	df = pd.DataFrame(filtered_data)
	df['targets'] = df['target_list'].apply(lambda x: '\n'.join(x))
	df['corrected_targets'] = np.nan
	df['is_target_corrected'] = False
	df['notes'] = np.nan
	stored_path = f'temp/data_cleaning/{lang}/{dataset_folder}/train.csv'
	os.makedirs(os.path.dirname(stored_path), exist_ok=True)
	df[['sentence_id', 'element_order', 'input', 'targets', 'corrected_targets', 'is_target_corrected', 'notes']].to_csv(stored_path, index=False)

Loaded 12410 records from hotel_dataset/eng/clean_traintruncated/hotel_aste_train_augmented_noreasoning.json
Loaded 12410 records from hotel_dataset/indo/clean_traintruncated/hotel_aste_train_augmented_noreasoning.json
Loaded 12410 records from hotel_dataset/sunda/clean_traintruncated/hotel_aste_train_augmented_noreasoning.json


In [8]:
df

,sentence_id,instance_id,task_elements,input,target,element_order,target_list
0,0,0,aos,my room had a problem with the ac not function...,[A] ac [O] not functioning optimally [S] negat...,aos,[[A] ac [O] not functioning optimally [S] nega...
1,1,5,aos,the place is nice . the swimming pool is clean...,[A] place [O] nice [S] positive [SSEP] [A] swi...,aos,"[[A] place [O] nice [S] positive, [A] swimming..."
2,2,10,aos,"it's really good , but the ac temperature cann...",[A] ac [O] cannot be adjusted [S] negative [SS...,aos,"[[A] ac [O] cannot be adjusted [S] negative, [..."
3,3,15,aos,cool . everything is comfortable . [A] [O] [S],[A] everything [O] comfortable [S] positive [S...,aos,"[[A] everything [O] comfortable [S] positive, ..."
4,4,20,aos,did not receive a snack . only after complaini...,[A] snack [O] did not receive [S] negative,aos,[[A] snack [O] did not receive [S] negative]
...,...,...,...,...,...,...,...
2477,2495,12475,aos,the wifi is not strong enough . [A] [O] [S],[A] wifi [O] not strong enough [S] negative,aos,[[A] wifi [O] not strong enough [S] negative]
2478,2496,12480,aos,"the room is quite clean , just cramped . [A] [...",[A] room [O] quite clean [S] positive [SSEP] [...,aos,"[[A] room [O] quite clean [S] positive, [A] ro..."
2479,2497,12485,aos,"comfortable , clean , and the service is very ...",[A] service [O] very friendly [S] positive [SS...,aos,"[[A] service [O] very friendly [S] positive, [..."
2480,2498,12490,aos,very disappointed with the room and the staff ...,[A] room [O] very disappointed [S] negative [S...,aos,"[[A] room [O] very disappointed [S] negative, ..."
